# IQ-235 — full source-checkpoint TPU inference
Select **Runtime → Change runtime type → TPU**. This runs the actual 235-layer model with BF16 expert streaming. The stored checkpoint uses the Mamba/Llama lexical foundation, not GLM-5.3 token IDs. No paid resources or training are launched.

In [ ]:
from pathlib import Path
import subprocess
repo = Path("/content/IQ")
if not repo.exists():
    subprocess.run(["git", "clone", "-b", "feat/iq235-colab-tpu-streaming",
                    "https://github.com/Harqer/IQ.git", str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only",
                    "origin", "feat/iq235-colab-tpu-streaming"], check=True)
%cd /content/IQ
%pip install 'huggingface_hub==1.20.1' 'transformers==5.17.0' safetensors

Use the Colab runtime's native TorchTPU installation. Do not install torch_xla or upgrade torch independently. If you previously changed the accelerator packages, use a fresh Colab TPU runtime if needed. Sign in to Hugging Face below using browser OAuth; no Colab secret or manually copied access token is needed. Your account must have access to the weight bucket and official source tokenizer for text mode.

In [ ]:
from huggingface_hub import login, HfApi, get_token
if not get_token():
    login(add_to_git_credential=False)
print("Signed in as:", HfApi().whoami(token=True)["name"])

## Optional: direct mounted checkpoint
If hf-mount is installed in the runtime, run the next cell before inference. It uses a read-only NFS mount by default. On Colab, mount permissions are not guaranteed. If the mount fails, the inference cell uses the bucket API. No full checkpoint download is required in either mode.

In [ ]:
import os, shutil, subprocess
from pathlib import Path
from huggingface_hub import get_token

def mount_environment():
    token = get_token()
    if not token:
        raise RuntimeError("Run the Hugging Face sign-in cell first")
    # hf-mount requires a bearer credential; use the active OAuth login
    # for this child process only, never print it or place it in arguments.
    return dict(os.environ, HF_TOKEN=token)
MOUNT = Path("/content/iq-weights")
RUN = MOUNT / "IQ/runs/glm53-full78-local-20261025a"
if shutil.which("hf-mount"):
    MOUNT.mkdir(exist_ok=True)
    if not (RUN / "iq_config.json").is_file():
        subprocess.run(["hf-mount", "start", "--read-only", "--cache-size",
                        "10000000000", "bucket", "kofecheckmate/iq-glm53-all256",
                        str(MOUNT)], env=mount_environment(), check=False)
    print("Mount visible:", (RUN / "iq_config.json").is_file())
else:
    print("hf-mount not installed; see docs/IQ235_COLAB_TPU.md for installation.")

In [ ]:
%cd /content/IQ
import subprocess, sys
from pathlib import Path
from huggingface_hub import HfApi
HfApi().whoami(token=True)
# Explicit full-model token-ID mode: no gated tokenizer download.
# Set TEXT_MODE=True only after obtaining authorized source tokenizer access.
TEXT_MODE = False
TOKENIZER_DIR = None  # Optional authorized local tokenizer directory.
cmd = [sys.executable, "-u", "scripts/iq235_colab_tpu.py",
       "--device", "tpu", "--max-new-tokens", "2"]
if TEXT_MODE:
    cmd += ["--prompt", "The capital of France is"]
    if TOKENIZER_DIR:
        cmd += ["--tokenizer-dir", str(TOKENIZER_DIR)]
else:
    cmd += ["--token-ids", "128000", "32"]
mount = Path("/content/iq-weights")
run = mount / "IQ/runs/glm53-full78-local-20261025a"
if (run / "iq_config.json").is_file():
    cmd += ["--mount-dir", str(mount)]
print("Mode:", "text" if TEXT_MODE else "full-model token IDs (no decoded text)")
print("Transport:", "hf-mount" if "--mount-dir" in cmd else "bucket API")
subprocess.run(cmd, check=True)

The token-ID mode runs the same 235 layers and generates actual token IDs, but does not validate decoded language. For text generation, obtain access to the original tokenizer or provide an authorized local copy, then enable TEXT_MODE above. In the reported Colab environment, torch 2.15.0+cu132 with libtpu 0.0.53 passed the native TPU BF16 probe; do not upgrade these packages if the probe works.